# Modeling: Artist Genre Clustering & Feature Analysis

This notebook demonstrates three modeling concepts applied to the Spotify dataset:
1. **Feature Engineering** — build a genre feature matrix from track data
2. **Hyperparameter Tuning** — find optimal k using silhouette score + RandomizedSearchCV
3. **Feature Importance** — identify which genres most distinguish clusters using Random Forest

## 0. Setup

In [ ]:
import polars as pl
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.cluster import KMeans
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import sys
sys.path.append('..')
import data.data_loader as data_loader

print('Libraries loaded.')

## 1. Load Data

In [ ]:
spotify_df = data_loader.load_spotify_data(limit=10000)

print(f'Spotify tracks: {spotify_df.height}')
print(f'Columns: {spotify_df.columns}')
spotify_df.head(3)

## 2. Feature Engineering: Build Artist-Genre Matrix

We cluster **artists** by their genre profiles. For each artist we collect all genres
associated with their tracks, then one-hot encode them using `MultiLabelBinarizer`.

This transforms raw categorical genre lists into a numeric binary matrix suitable for ML.

In [ ]:
# explode artists and aggregate genres per artist
artist_genre_df = (
    spotify_df
    .with_columns(
        pl.col('artists').str.split(';').alias('artists_list')
    )
    .explode('artists_list')
    .with_columns(pl.col('artists_list').str.strip_chars().alias('artist'))
    .group_by('artist')
    .agg(
        pl.col('track_genre').unique().alias('genres'),
        pl.col('popularity').mean().round(1).alias('avg_popularity'),
        pl.col('track_id').n_unique().alias('track_count')
    )
    .filter(pl.col('track_count') >= 2)
    .sort('avg_popularity', descending=True)
)

print(f'Artists with >= 2 tracks: {artist_genre_df.height}')
artist_genre_df.head(5)

In [ ]:
# one-hot encode genre lists into binary matrix
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(artist_genre_df['genres'].to_list())
genre_classes = mlb.classes_

print(f'Feature matrix shape: {genre_matrix.shape}')
print(f'Number of unique genres: {len(genre_classes)}')
print(f'Sample genres: {list(genre_classes[:10])}')

In [ ]:
# visualize top genres by artist count
genre_counts = genre_matrix.sum(axis=0)
genre_freq_df = pd.DataFrame({
    'genre': genre_classes,
    'artist_count': genre_counts
}).sort_values('artist_count', ascending=False).head(20)

fig = px.bar(
    genre_freq_df,
    x='genre', y='artist_count',
    title='Top 20 Genres by Number of Artists',
    labels={'genre': 'Genre', 'artist_count': 'Number of Artists'},
    color='artist_count',
    color_continuous_scale='Viridis'
)
fig.update_layout(xaxis_tickangle=45, height=450)
fig.show()

## 3. Hyperparameter Tuning: Find Optimal K

Rather than guessing k, we sweep k=2..10 and score each with **silhouette score**.
We also plot the **elbow curve** (inertia) as a secondary check.
The best k is selected objectively from the silhouette scores.

In [ ]:
# PCA for faster, more stable clustering
n_components = min(20, genre_matrix.shape[1] - 1)
pca = PCA(n_components=n_components, random_state=42)
genre_matrix_pca = pca.fit_transform(genre_matrix)

print(f'Variance explained by {n_components} PCA components: {pca.explained_variance_ratio_.sum():.2%}')

k_values = list(range(2, 11))
silhouette_scores = []
inertias = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(genre_matrix_pca)
    sil = silhouette_score(genre_matrix_pca, labels)
    silhouette_scores.append(sil)
    inertias.append(kmeans.inertia_)
    print(f'k={k}: silhouette={sil:.4f}, inertia={kmeans.inertia_:.1f}')

best_k = k_values[int(np.argmax(silhouette_scores))]
print(f'\nBest k: {best_k} (silhouette={max(silhouette_scores):.4f})')

In [ ]:
# plot silhouette + elbow side by side
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Silhouette Score by k (higher = better)',
        'Elbow Curve (lower inertia = better)'
    ]
)

fig.add_trace(
    go.Scatter(
        x=k_values, y=silhouette_scores,
        mode='lines+markers',
        marker=dict(
            color=['red' if k == best_k else '#667eea' for k in k_values],
            size=10
        ),
        name='Silhouette'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=k_values, y=inertias,
        mode='lines+markers',
        marker=dict(color='#764ba2', size=8),
        name='Inertia'
    ),
    row=1, col=2
)

fig.update_xaxes(title_text='Number of Clusters (k)')
fig.update_layout(
    height=400,
    title_text=f'Hyperparameter Tuning: Optimal k={best_k}',
    showlegend=False
)
fig.show()

## 4. K-Means Clustering with Optimal K

In [ ]:
# fit final model
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(genre_matrix_pca)

# attach labels to artist dataframe
artist_clusters_df = artist_genre_df.with_columns(
    pl.Series('cluster', cluster_labels)
).to_pandas()
artist_clusters_df['genres_str'] = artist_clusters_df['genres'].apply(
    lambda x: ', '.join(sorted(x)) if isinstance(x, list) else str(x)
)

cluster_counts = artist_clusters_df.groupby('cluster').size().reset_index(name='artist_count')
print('Artists per cluster:')
print(cluster_counts.to_string(index=False))

In [ ]:
# 2D PCA scatter plot of clusters
pca_2d = PCA(n_components=2, random_state=42)
coords = pca_2d.fit_transform(genre_matrix)

viz_df = pd.DataFrame({
    'pc1': coords[:, 0],
    'pc2': coords[:, 1],
    'cluster': cluster_labels.astype(str),
    'artist': artist_genre_df['artist'].to_list(),
    'avg_popularity': artist_genre_df['avg_popularity'].to_list(),
    'genres': artist_clusters_df['genres_str'].to_list()
})

fig = px.scatter(
    viz_df, x='pc1', y='pc2',
    color='cluster',
    title=f'Artist Genre Clusters (k={best_k}) — PCA 2D Projection',
    labels={'pc1': 'PC 1', 'pc2': 'PC 2'},
    hover_data=['artist', 'avg_popularity', 'genres'],
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_layout(height=520)
fig.show()

In [ ]:
# summarize top genres and sample artists per cluster
genre_df = pd.DataFrame(genre_matrix, columns=genre_classes)
genre_df['cluster'] = cluster_labels
genre_df['artist'] = artist_genre_df['artist'].to_list()
genre_df['avg_popularity'] = artist_genre_df['avg_popularity'].to_list()

print(f'Top genres and most popular artists per cluster:\n')
for c in sorted(genre_df['cluster'].unique()):
    cluster_data = genre_df[genre_df['cluster'] == c]
    top_genres = (
        cluster_data
        .drop(columns=['cluster', 'artist', 'avg_popularity'])
        .mean()
        .sort_values(ascending=False)
        .head(5)
    )
    top_artists = cluster_data.nlargest(3, 'avg_popularity')['artist'].tolist()
    genre_str = ', '.join([f"{g} ({v:.2f})" for g, v in top_genres.items()])
    print(f'Cluster {c} ({len(cluster_data)} artists)')
    print(f'  Top genres:  {genre_str}')
    print(f'  Top artists: {", ".join(top_artists)}')
    print()

## 5. Feature Importance: Which Genres Distinguish Clusters?

We train a **Random Forest classifier** to predict cluster membership from genre features.
The Gini-based feature importances reveal which genres most strongly define each cluster.
We also run **RandomizedSearchCV** to tune the Random Forest hyperparameters.

In [ ]:
# baseline random forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(genre_matrix, cluster_labels)

cv_scores = cross_val_score(rf, genre_matrix, cluster_labels, cv=5, scoring='accuracy')
print(f'Baseline RF CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

In [ ]:
# hyperparameter tuning with RandomizedSearchCV
param_dist = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', 0.3],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='accuracy',
    random_state=42,
    verbose=1
)
rf_search.fit(genre_matrix, cluster_labels)

print(f'\nBest params:      {rf_search.best_params_}')
print(f'Best CV accuracy: {rf_search.best_score_:.3f}')
print(f'Improvement:      +{rf_search.best_score_ - cv_scores.mean():.3f} over baseline')

In [ ]:
# feature importances from tuned model
best_rf = rf_search.best_estimator_

importances_df = pd.DataFrame({
    'genre': genre_classes,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False).head(20)

fig = px.bar(
    importances_df,
    x='importance', y='genre',
    orientation='h',
    title='Top 20 Genre Feature Importances (Tuned Random Forest)',
    labels={'importance': 'Feature Importance (Gini)', 'genre': 'Genre'},
    color='importance',
    color_continuous_scale='Blues'
)
fig.update_layout(height=560, yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# baseline vs tuned importances side by side
compare_df = pd.DataFrame({
    'genre': genre_classes,
    'baseline': rf.feature_importances_,
    'tuned': best_rf.feature_importances_
}).sort_values('tuned', ascending=False).head(15)

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Baseline RF',
    y=compare_df['genre'], x=compare_df['baseline'],
    orientation='h', marker_color='#667eea'
))
fig.add_trace(go.Bar(
    name='Tuned RF',
    y=compare_df['genre'], x=compare_df['tuned'],
    orientation='h', marker_color='#764ba2'
))
fig.update_layout(
    barmode='group',
    title='Baseline vs Tuned Random Forest: Top Genre Importances',
    xaxis_title='Importance (Gini)',
    yaxis=dict(categoryorder='total ascending'),
    height=500
)
fig.show()

## 6. Summary

| Concept | What we did | Key result |
|---|---|---|
| **Feature Engineering** | Multi-label binarized artist genre profiles into binary matrix | `(n_artists, n_genres)` feature matrix |
| **Hyperparameter Tuning (K-Means)** | Silhouette score sweep over k=2..10 | Objectively selected best k |
| **Clustering** | K-Means on PCA-reduced genre matrix | Artists grouped into genre-coherent clusters |
| **Feature Importance** | Random Forest trained to predict cluster membership | Top genres defining each cluster identified |
| **Hyperparameter Tuning (RF)** | RandomizedSearchCV over depth/estimators/features | Improved accuracy over baseline |